The purpose of this notebook is to verify the correctness of the functions in `utils_hamiltonian.py`, and present simple examples of their usage

In [26]:
import numpy as np
from utils_hamiltonian import *
from openfermion import *
from openfermionpyscf import *
from utils_basic import random_pauli_term, random_bin_list, state_from_bin_list
from numpy.random import uniform
import random


## Pauli Operator Matrix Elements

Here, I verify the following:
- `split_pauli_operator` should cleave a Pauli operator into two Pauli operators correctly


In [2]:
P = QubitOperator('X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9')
N = 5
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


P = QubitOperator('X0 Y5 Y6 Y7 Y8 Y9')
N = 3
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


    Original Pauli : 1.0 [X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9]
    Left Split     : 1.0 [X0 X1 X2 X3 X4]
    Right Split    : 1.0 [Y5 Y6 Y7 Y8 Y9]


    Original Pauli : 1.0 [X0 Y5 Y6 Y7 Y8 Y9]
    Left Split     : 1.0 [X0]
    Right Split    : 1.0 [Y5 Y6 Y7 Y8 Y9]



Here, I verify the following:

- <v|P|w> is calculated correctly for a bunch of P, v, w examples by comparing with numerical value

In [3]:
for Nqubits in range(2, 12):
    for _ in range(40):
        _, P = random_pauli_term(Nqubits)
        v    = random_bin_list(Nqubits)
        w    = random_bin_list(Nqubits)

        matrix_element_numerical  = state_from_bin_list(v) @ get_sparse_operator(P, Nqubits) @ state_from_bin_list(w)
        matrix_element_analytical = pauli_matrix_element_with_basis_state(P, v, w)
        
        print(np.abs(matrix_element_numerical - matrix_element_analytical) < 1e-12, end='')
    print()

TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTr

## Hamiltonian modification based on symmetries

First, I generate a simple non-trivial Hamiltonian to work with

In [11]:
geo = [
    ['H', [0,0,0]],
    ['H', [0,0,1]],
    ['H', [0,0,2]],
    ['H', [0,0,3]]
]
mol = run_pyscf(
    MolecularData(geo, 'sto3g', 1, 0), 
    run_scf=True
)
H = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))

# play around with this example by modifying v and w
v = [0,1,1,0]
w = [1,0,0,1]

Hrel = remove_irrelevant_pauli_terms(H, v, w)

print('Hrel')
print(Hrel)
print()

Htapered = taper_hamiltonian(H, v, w)

print('Htapered')
print(Htapered)

print()
print(Htapered - taper_hamiltonian(Hrel, v, w))

Hrel
(-0.039345513560407545+0j) [X0 X1 Y2 Y3] +
(0.039345513560407545+0j) [X0 Y1 Y2 X3] +
(0.039345513560407545+0j) [Y0 X1 X2 Y3] +
(-0.039345513560407545+0j) [Y0 Y1 X2 X3]

Htapered
(-0.15738205424163018+0j) []

0


The purpose of this code is to modify the Hamiltonian without changing the relevant matrix element and/or expectation values. So we should see if that works as desired.

Here, I see if, for states $|\Psi_s\rangle = |s\rangle |\psi_s\rangle$, and $|\Psi_t\rangle = |t\rangle |\psi_t\rangle$, we have

$$
\langle \Psi_s|H|\Psi_t\rangle = \langle \Psi_s|H_{\text{rel}}|\Psi_t\rangle = \langle \psi_s|H_\text{tapered}|\psi_t \rangle
$$

I will use the LiH Hamiltonian since it is larger

In [28]:
geo = [
    ['H', [0,0,0]],
    ['Li', [0,0,1]]
]
mol     = run_pyscf(MolecularData(geo, 'sto3g', 1, 0), run_scf=True)
H       = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))
Nqubits = 12

for _ in range(20):
    taper_size = random.sample(range(2, 8), 1)[0]

    bin_s = random_bin_list(taper_size)
    bin_t = random_bin_list(taper_size)

    ket_s = state_from_bin_list(bin_s, taper_size)
    ket_t = state_from_bin_list(bin_t, taper_size)

    psi_s = uniform(-1, 1, 2 ** (Nqubits - taper_size))
    psi_t = uniform(-1, 1, 2 ** (Nqubits - taper_size))

    Psi_s = np.kron(ket_s, psi_s)
    Psi_t = np.kron(ket_t, psi_t)


    Hsparse         = get_sparse_operator(H, Nqubits)
    ev_full         = Psi_s @ Hsparse @ Psi_t

    Hrel_sparse     = get_sparse_operator(remove_irrelevant_pauli_terms(H, bin_s, bin_t), Nqubits)
    ev_rel          = Psi_s @ Hrel_sparse @ Psi_t

    Htapered_sparse = get_sparse_operator(taper_hamiltonian(H, bin_s, bin_t), Nqubits - taper_size)
    ev_tapered      = psi_s @ Htapered_sparse @ psi_t

    print(f'''
        full expectation    : {ev_full}
        rel expectation     : {ev_rel}
        tapered expectation : {ev_tapered}

        all equal           : {np.abs(ev_full-ev_rel)<1e-12 and np.abs(ev_full-ev_tapered)<1e-12 and np.abs(ev_rel-ev_tapered)<1e-12}
    ''')


        full expectation    : (0.02482657678400841+0j)
        rel expectation     : (0.02482657678400841+0j)
        tapered expectation : (0.02482657678400841+0j)

        all equal           : True
    

        full expectation    : (-0.10104624802798683+0j)
        rel expectation     : (-0.10104624802798683+0j)
        tapered expectation : (-0.10104624802798683+0j)

        all equal           : True
    

        full expectation    : 0j
        rel expectation     : 0j
        tapered expectation : 0.0

        all equal           : True
    

        full expectation    : 0j
        rel expectation     : 0j
        tapered expectation : 0.0

        all equal           : True
    

        full expectation    : (0.016380169124239397+0j)
        rel expectation     : (0.016380169124239397+0j)
        tapered expectation : (0.016380169124239397+0j)

        all equal           : True
    

        full expectation    : 0j
        rel expectation     : 0.0
        tapered expec

3